# 02 - Eksperimen & Komparasi 5 Model (*From Scratch* vs `scikit-learn`)
**Penanggung Jawab:** Kolektif Seluruh Anggota (Anggota 1 - 5)  
**Kelompok:** G15 - MessiLearning (IF3170 Inteligensi Artifisial)

---
Notebook ini berisi pengujian 5 algoritma Supervised Learning *from scratch* menggunakan data sintetis (selama fase paralel) maupun data bersih hasil `DataPipeline`, lengkap dengan komparasi metrik (Akurasi, Presisi, Recall, F1-Score, Waktu Eksekusi) terhadap `scikit-learn`.

In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath('..'))

from src.utils import (
    set_global_seed,
    RANDOM_SEED,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    calculate_custom_cost,
    k_fold_split,
)
from src.dummy_generator import generate_clean_synthetic_data, generate_dtl_synthetic_data
from src.knn import KNN
from src.dtl import DecisionTree, export_tree_visualization
from src.logistic_regression import LogisticRegression, generate_loss_contour_video
from src.svm import SVM, plot_svm_margin_2d
from src.naive_bayes import NaiveBayes, export_bayesian_network_graph

set_global_seed(RANDOM_SEED)
X_dummy, y_dummy = generate_clean_synthetic_data(n_samples=500, n_features=6)
X_dtl_dummy, y_dtl_dummy = generate_dtl_synthetic_data(n_samples=500)
print(f"Dummy Clean Shape: {X_dummy.shape}, Dummy DTL Shape: {X_dtl_dummy.shape}")

## 1. K-Nearest Neighbors (KNN) — Anggota 1
Komparasi `src.knn.KNN` vs `sklearn.neighbors.KNeighborsClassifier`

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
# TODO (Anggota 1): Evaluasi KNN from scratch vs scikit-learn

## 2. Decision Tree Learning (DTL) — Anggota 2 (`13524053`)

Implementasi **Decision Tree Learning *from scratch*** pada `src/dtl.py` (100% NumPy untuk algoritma inti) mencakup:
1. **Spesifikasi Wajib**: Mendukung ketiga varian (`ID3`, `C4.5`, `CART`), penanganan fitur numerik kontinu (*midpoint threshold*), fitur kategorikal (*multi-way* ID3/C4.5 & *binary* CART), penanganan *missing values* (`NaN`) ala Quinlan C4.5 (*scaled Gain* & *fractional weights*), *save/load*, serta komparasi terhadap `sklearn.tree.DecisionTreeClassifier` pada **dataset penginapan asli**.
2. **Bonus Visualisasi Tree (+1 Poin)**: Fungsi `export_tree_visualization(model, top_n=3, ...)` yang mengekspor struktur percabangan pohon hingga kedalaman `top_n` ke `output/plots/dtl_tree_top_n.png`.
3. **Bonus Optimasi Model DTL (+2 Poin — Di Luar Salindia Materi 10a/10c)**:
   - **Histogram-Based Quantile Split Optimization** (*Ke et al., 2017 [5]; Chen & Guestrin, 2016 [6]*): Mengganti pencarian *split* atribut kontinu *exact* $O(N \log N)$ dengan pra-diskretisasi kuantil $B$ *bins* (dihitung hanya pada data latih per *fold* agar bebas *data leakage*) dan akumulasi histogram $O(N)$ + pencarian *split* $O(B)$, sehingga mempercepat pelatihan secara signifikan sekaligus bertindak sebagai regularisasi terhadap *noise* numerik.
   - **Minimal Cost-Complexity Weakest-Link Pruning Path** (*Breiman et al., 1984 [4], Bab 3*): Meminimalkan $R_\alpha(T) = R(T) + \alpha |L(T)|$ berbasis *weighted impurity* menggunakan *min-heap weakest-link* tanpa mengorbankan 15–20% data latih untuk *pruning set* (dibandingkan terhadap *baseline* salindia kuliah: *Exact Split + Reduced Error Pruning* dari Materi 10c).

**Referensi Ilmiah (IEEE):**
- `[1]` J. R. Quinlan, "Induction of decision trees," *Machine Learning*, vol. 1, no. 1, pp. 81–106, 1986.
- `[2]` J. R. Quinlan, *C4.5: Programs for Machine Learning*. San Mateo, CA, USA: Morgan Kaufmann, 1993.
- `[3]` T. M. Mitchell, *Machine Learning*. New York, NY, USA: McGraw-Hill, 1997, ch. 3.
- `[4]` L. Breiman, J. H. Friedman, R. A. Olshen, and C. J. Stone, *Classification and Regression Trees*. Wadsworth, 1984, ch. 3.
- `[5]` G. Ke *et al.*, "LightGBM: A highly efficient gradient boosting decision tree," in *Advances in Neural Information Processing Systems (NeurIPS)*, vol. 30, 2017, pp. 3146–3154.
- `[6]` T. Chen and C. Guestrin, "XGBoost: A scalable tree boosting system," in *Proc. 22nd ACM SIGKDD Int. Conf. Knowledge Discovery and Data Mining (KDD)*, 2016, pp. 785–794.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from scipy.stats import chi2
from src.dtl import DecisionTree, export_tree_visualization
from src.utils import confusion_matrix

root_dir = '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'
dtl_proc_path = os.path.join(root_dir, 'data', 'processed', 'train_dtl.csv')
dtl_raw_path = os.path.join(root_dir, 'data', 'raw', 'train.csv')

def macro_f1_score(y_true, y_pred):
    return 0.5 * (f1_score(y_true, y_pred) + f1_score(1 - y_true, 1 - y_pred))

def mcnemar_test(y_true, pred_a, pred_b):
    """Uji signifikansi McNemar (Materi 13) dengan koreksi kontinuitas Edwards (df=1)."""
    correct_a = (pred_a == y_true)
    correct_b = (pred_b == y_true)
    b = int(np.sum(correct_a & ~correct_b))
    c = int(np.sum(~correct_a & correct_b))
    stat = ((abs(b - c) - 1.0) ** 2) / (b + c) if (b + c) > 0 else 0.0
    p_val = float(1.0 - chi2.cdf(stat, df=1)) if (b + c) > 0 else 1.0
    return b, c, stat, p_val

# 1. Muat Dataset Penginapan Asli (prioritas: processed/train_dtl.csv -> raw/train.csv -> synthetic fallback)
if os.path.exists(dtl_proc_path):
    df_dtl = pd.read_csv(dtl_proc_path, dtype={'id': str})
    src_label = dtl_proc_path
elif os.path.exists(dtl_raw_path):
    df_dtl = pd.read_csv(dtl_raw_path, dtype={'id': str})
    src_label = dtl_raw_path
else:
    X_syn, y_syn = generate_dtl_synthetic_data(n_samples=600, random_state=RANDOM_SEED)
    df_dtl = pd.DataFrame(X_syn, columns=['lead_time', 'adr_price', 'market_segment', 'deposit_type'])
    df_dtl['is_canceled'] = y_syn
    src_label = 'synthetic_fallback'

target_col = 'is_canceled' if 'is_canceled' in df_dtl.columns else df_dtl.columns[-1]
dtl_feat_names = [c for c in df_dtl.columns if c not in ('id', target_col)]
cat_col_names = {'meal_package', 'room_category', 'booking_channel', 'market_segment', 'deposit_type'}
dtl_cat_indices = [i for i, c in enumerate(dtl_feat_names) if c in cat_col_names or df_dtl[c].dtype == object or str(df_dtl[c].dtype) in ('str', 'string')]

X_dtl_eval = df_dtl[dtl_feat_names].to_numpy(dtype=object)
y_dtl_eval = df_dtl[target_col].to_numpy(dtype=int)
n_missing_total = int(df_dtl[dtl_feat_names].isna().sum().sum())
print(f"[DTL] Sumber data : {src_label}")
print(f"[DTL] Dimensi     : {X_dtl_eval.shape} | Total sel NaN: {n_missing_total:,} | Fitur kategorikal: {[dtl_feat_names[i] for i in dtl_cat_indices]}")

# =============================================================================
# EKSPERIMEN 1: ABLATION STUDY BONUS 6 (5-Fold CV Bebas Data Leakage)
# Membandingkan Baseline Salindia Kuliah (Exact + REP) vs Bonus 6 (Histogram Quantile Split + Minimal CCP)
# =============================================================================
ablation_configs = [
    ('Baseline Kuliah: C4.5 Exact (Unpruned)', dict(criterion='c45', split_strategy='exact', max_depth=18, min_samples_leaf=5, use_pruning=False)),
    ('Baseline Kuliah: C4.5 Exact + REP (Materi 10c)', dict(criterion='c45', split_strategy='exact', max_depth=18, min_samples_leaf=5, use_pruning=True, pruning_method='rep', val_split_ratio=0.15)),
    ('Bonus 6: C4.5 Exact + Minimal CCP (α=0.0003)', dict(criterion='c45', split_strategy='exact', max_depth=18, min_samples_leaf=5, use_pruning=True, pruning_method='ccp', ccp_alpha=0.0003)),
    ('Bonus 6: C4.5 Hist (B=16) + Minimal CCP', dict(criterion='c45', split_strategy='histogram', max_bins=16, max_depth=18, min_samples_leaf=5, use_pruning=True, pruning_method='ccp', ccp_alpha=0.0003)),
    ('Bonus 6: C4.5 Hist (B=32) + Minimal CCP', dict(criterion='c45', split_strategy='histogram', max_bins=32, max_depth=18, min_samples_leaf=5, use_pruning=True, pruning_method='ccp', ccp_alpha=0.0003)),
    ('Bonus 6: C4.5 Hist (B=64) + Minimal CCP', dict(criterion='c45', split_strategy='histogram', max_bins=64, max_depth=18, min_samples_leaf=5, use_pruning=True, pruning_method='ccp', ccp_alpha=0.0003)),
    ('Bonus 6: C4.5 Hist (B=128) + Minimal CCP', dict(criterion='c45', split_strategy='histogram', max_bins=128, max_depth=18, min_samples_leaf=5, use_pruning=True, pruning_method='ccp', ccp_alpha=0.0003)),
]

ablation_records = []
oof_preds_store = {}
for name, params in ablation_configs:
    mf1s, accs, precs, recs, costs, nodes_list, depths_list, times_s = [], [], [], [], [], [], [], []
    oof_pred = np.zeros(len(y_dtl_eval), dtype=int)
    for train_idx, val_idx in k_fold_split(X_dtl_eval, y_dtl_eval, k=5, seed=RANDOM_SEED):
        t0 = time.perf_counter()
        clf = DecisionTree(**params, categorical_features=dtl_cat_indices, random_state=RANDOM_SEED)
        clf.fit(X_dtl_eval[train_idx], y_dtl_eval[train_idx])
        preds = clf.predict(X_dtl_eval[val_idx])
        elapsed = time.perf_counter() - t0
        oof_pred[val_idx] = preds
        mf1s.append(macro_f1_score(y_dtl_eval[val_idx], preds))
        accs.append(accuracy_score(y_dtl_eval[val_idx], preds))
        precs.append(precision_score(y_dtl_eval[val_idx], preds))
        recs.append(recall_score(y_dtl_eval[val_idx], preds))
        costs.append(calculate_custom_cost(y_dtl_eval[val_idx], preds))
        nodes_list.append(clf.get_n_nodes())
        depths_list.append(clf.get_depth())
        times_s.append(elapsed)
    oof_preds_store[name] = oof_pred
    ablation_records.append({
        'Konfigurasi DTL (5-Fold CV)': name,
        'Macro F1 (Mean)': np.mean(mf1s),
        'Macro F1 (Std)': np.std(mf1s),
        'Accuracy': np.mean(accs),
        'Precision': np.mean(precs),
        'Recall': np.mean(recs),
        'Avg Cost (2FP+FN)': np.mean(costs),
        'Avg Nodes': np.mean(nodes_list),
        'Avg Depth': np.mean(depths_list),
        'Avg Time (s)': np.mean(times_s),
    })

df_dtl_ablation = pd.DataFrame(ablation_records)
print("\n=== TABEL 1: ABLATION STUDY BONUS 6 (BASELINE KULIAH VS HISTOGRAM QUANTILE + MINIMAL CCP) ===")
display(df_dtl_ablation.round(4))

# Uji McNemar antara Baseline Kuliah (C4.5 Exact + REP) vs Bonus 6 (C4.5 Hist B=64 + Minimal CCP)
b_mc, c_mc, stat_mc, p_mc = mcnemar_test(
    y_dtl_eval,
    oof_preds_store['Baseline Kuliah: C4.5 Exact + REP (Materi 10c)'],
    oof_preds_store['Bonus 6: C4.5 Hist (B=64) + Minimal CCP'],
)
print(f"[McNemar Test — Baseline REP vs Bonus 6 Hist+CCP] b={b_mc}, c={c_mc}, chi2={stat_mc:.4f}, p-value={p_mc:.6f}")

# =============================================================================
# EKSPERIMEN 2: KOMPARASI HEAD-TO-HEAD FROM SCRATCH VS SCIKIT-LEARN (DATASET ASLI)
# Parameter wajib Spesifikasi Hal. 10: ID3='entropy', C4.5='entropy'+ccp_alpha, CART='gini'
# =============================================================================
def prepare_sklearn_fold(df_tr, df_va, feat_cols, cat_idxs):
    """Imputasi median/modus & ordinal encoding khusus sklearn HANYA dari fold training (bebas leakage)."""
    X_tr_s = np.empty((len(df_tr), len(feat_cols)), dtype=float)
    X_va_s = np.empty((len(df_va), len(feat_cols)), dtype=float)
    cat_set = set(cat_idxs)
    for j, col in enumerate(feat_cols):
        s_tr = df_tr[col]
        s_va = df_va[col]
        if j in cat_set:
            mode_val = s_tr.mode(dropna=True).iloc[0] if not s_tr.mode(dropna=True).empty else '__MISS__'
            tr_fill = s_tr.fillna(mode_val).astype(str)
            va_fill = s_va.fillna(mode_val).astype(str)
            vocab = {v: idx for idx, v in enumerate(tr_fill.unique())}
            X_tr_s[:, j] = tr_fill.map(vocab).to_numpy(dtype=float)
            X_va_s[:, j] = va_fill.map(vocab).fillna(-1.0).to_numpy(dtype=float)
        else:
            med = float(pd.to_numeric(s_tr, errors='coerce').median())
            X_tr_s[:, j] = pd.to_numeric(s_tr, errors='coerce').fillna(med).to_numpy(dtype=float)
            X_va_s[:, j] = pd.to_numeric(s_va, errors='coerce').fillna(med).to_numpy(dtype=float)
    return X_tr_s, X_va_s

sklearn_comp_configs = [
    ('Scratch ID3 (Histogram B=64, max_depth=18)', True, lambda: DecisionTree(criterion='id3', split_strategy='histogram', max_bins=64, max_depth=18, min_samples_leaf=5, categorical_features=dtl_cat_indices, random_state=RANDOM_SEED)),
    ('Sklearn ID3 (criterion=entropy, max_depth=18)', False, lambda: DecisionTreeClassifier(criterion='entropy', max_depth=18, min_samples_leaf=5, random_state=RANDOM_SEED)),
    ('Scratch C4.5 + Hist B=64 + CCP (α=0.0003)', True, lambda: DecisionTree(criterion='c45', split_strategy='histogram', max_bins=64, max_depth=18, min_samples_leaf=5, use_pruning=True, pruning_method='ccp', ccp_alpha=0.0003, categorical_features=dtl_cat_indices, random_state=RANDOM_SEED)),
    ('Sklearn C4.5 (criterion=entropy, ccp_alpha=0.0003)', False, lambda: DecisionTreeClassifier(criterion='entropy', ccp_alpha=0.0003, random_state=RANDOM_SEED)),
    ('Sklearn C4.5 (criterion=entropy, ccp_alpha=0.01)', False, lambda: DecisionTreeClassifier(criterion='entropy', ccp_alpha=0.01, random_state=RANDOM_SEED)),
    ('Scratch CART (Histogram B=64 + CCP α=0.0003)', True, lambda: DecisionTree(criterion='cart', split_strategy='histogram', max_bins=64, max_depth=18, min_samples_leaf=5, use_pruning=True, pruning_method='ccp', ccp_alpha=0.0003, categorical_features=dtl_cat_indices, random_state=RANDOM_SEED)),
    ('Sklearn CART (criterion=gini, max_depth=18)', False, lambda: DecisionTreeClassifier(criterion='gini', max_depth=18, min_samples_leaf=5, random_state=RANDOM_SEED)),
]

comp_records = []
for label, is_scratch, builder in sklearn_comp_configs:
    mf1s, accs, precs, recs, costs, times_s = [], [], [], [], [], []
    for train_idx, val_idx in k_fold_split(X_dtl_eval, y_dtl_eval, k=5, seed=RANDOM_SEED):
        clf = builder()
        if is_scratch:
            t0 = time.perf_counter()
            clf.fit(X_dtl_eval[train_idx], y_dtl_eval[train_idx])
            preds = clf.predict(X_dtl_eval[val_idx])
            elapsed = time.perf_counter() - t0
        else:
            X_tr_sk, X_va_sk = prepare_sklearn_fold(df_dtl.iloc[train_idx], df_dtl.iloc[val_idx], dtl_feat_names, dtl_cat_indices)
            t0 = time.perf_counter()
            clf.fit(X_tr_sk, y_dtl_eval[train_idx])
            preds = clf.predict(X_va_sk)
            elapsed = time.perf_counter() - t0
        mf1s.append(macro_f1_score(y_dtl_eval[val_idx], preds))
        accs.append(accuracy_score(y_dtl_eval[val_idx], preds))
        precs.append(precision_score(y_dtl_eval[val_idx], preds))
        recs.append(recall_score(y_dtl_eval[val_idx], preds))
        costs.append(calculate_custom_cost(y_dtl_eval[val_idx], preds))
        times_s.append(elapsed)
    comp_records.append({
        'Model Comparison (5-Fold CV Dataset Penginapan)': label,
        'Macro F1': np.mean(mf1s),
        'Macro F1 Std': np.std(mf1s),
        'Accuracy': np.mean(accs),
        'Precision': np.mean(precs),
        'Recall': np.mean(recs),
        'Avg Cost': np.mean(costs),
        'Avg Time (s)': np.mean(times_s),
    })

df_dtl_comp = pd.DataFrame(comp_records)
print("\n=== TABEL 2: KOMPARASI FROM SCRATCH VS SCIKIT-LEARN (DATASET PENGINAPAN ASLI) ===")
display(df_dtl_comp.round(4))

# =============================================================================
# EKSPERIMEN 3: SAVE/LOAD VERIFICATION, VISUALISASI TOP-N (+1 POIN), & ERROR ANALYSIS
# =============================================================================
n_holdout = int(0.2 * len(X_dtl_eval))
X_tr_fin, y_tr_fin = X_dtl_eval[:-n_holdout], y_dtl_eval[:-n_holdout]
X_te_fin, y_te_fin = X_dtl_eval[-n_holdout:], y_dtl_eval[-n_holdout:]

best_dtl_model = DecisionTree(
    criterion='c45',
    split_strategy='histogram',
    max_bins=64,
    max_depth=18,
    min_samples_leaf=5,
    use_pruning=True,
    pruning_method='ccp',
    ccp_alpha=0.0003,
    categorical_features=dtl_cat_indices,
    random_state=RANDOM_SEED,
)
best_dtl_model.feature_names_ = dtl_feat_names
best_dtl_model.fit(X_tr_fin, y_tr_fin)

model_out_path = os.path.join(root_dir, 'output', 'models', 'dtl_model.pkl')
plot_out_path = os.path.join(root_dir, 'output', 'plots', 'dtl_tree_top_n.png')
best_dtl_model.save(model_out_path)
loaded_dtl_model = DecisionTree.load(model_out_path)

# Verifikasi Save/Load identik 100%
proba_orig = best_dtl_model.predict_proba(X_te_fin)
proba_load = loaded_dtl_model.predict_proba(X_te_fin)
preds_fin = loaded_dtl_model.predict(X_te_fin)
assert np.allclose(proba_orig, proba_load), 'Probabilitas sebelum dan sesudah load berbeda!'
assert np.array_equal(best_dtl_model.predict(X_te_fin), preds_fin), 'Prediksi sebelum dan sesudah load berbeda!'
print(f"[Save/Load Verified] Prediksi & probabilitas 100% identik pada {len(X_te_fin):,} baris uji.")

# Ekspor Bonus Visualisasi Tree Top-3
export_tree_visualization(loaded_dtl_model, top_n=3, feature_names=dtl_feat_names, output_path=plot_out_path)

# Error Analysis singkat (Feature Importance, Confusion Matrix, & Performa pada baris NaN vs Lengkap)
tn, fp, fn, tp = confusion_matrix(y_te_fin, preds_fin)
has_nan_mask = df_dtl.iloc[-n_holdout:][dtl_feat_names].isna().any(axis=1).to_numpy()
mf1_nan = macro_f1_score(y_te_fin[has_nan_mask], preds_fin[has_nan_mask]) if np.any(has_nan_mask) else np.nan
mf1_complete = macro_f1_score(y_te_fin[~has_nan_mask], preds_fin[~has_nan_mask]) if np.any(~has_nan_mask) else np.nan
print(f"\n=== ERROR ANALYSIS (HOLDOUT 20%: {len(X_te_fin):,} BARIS) ===")
print(f"Confusion Matrix : TN={tn:,}, FP={fp:,}, FN={fn:,}, TP={tp:,}")
print(f"Holdout Macro F1 : {macro_f1_score(y_te_fin, preds_fin):.4f} | Accuracy: {accuracy_score(y_te_fin, preds_fin):.4f}")
print(f"  -> Baris mengandung NaN ({has_nan_mask.sum():,} baris): Macro F1 = {mf1_nan:.4f}")
print(f"  -> Baris lengkap tanpa NaN ({(~has_nan_mask).sum():,} baris): Macro F1 = {mf1_complete:.4f}")

df_imp = pd.DataFrame({'Feature': dtl_feat_names, 'Importance': loaded_dtl_model.feature_importances_}).sort_values('Importance', ascending=False)
print("\nTop-8 Fitur Paling Berpengaruh (Feature Importances DTL):")
display(df_imp.head(8).round(4))

## 3. Logistic Regression — Anggota 3
Komparasi `src.logistic_regression.LogisticRegression` ($L_1, L_2$, Elastic Net + Adam) vs `sklearn.linear_model.LogisticRegression` & Video Lintasan Loss

In [ ]:
from sklearn.linear_model import LogisticRegression as SklearnLogReg
# TODO (Anggota 3): Evaluasi LogisticRegression from scratch vs scikit-learn & render video kontur

## 4. Support Vector Machine (SVM) — Anggota 4
Komparasi `src.svm.SVM` (Linear, Polynomial, Gaussian RBF) vs `sklearn.svm.SVC` & Plot Bidang Pemisah 2D

In [ ]:
from sklearn.svm import SVC
# TODO (Anggota 4): Evaluasi SVM from scratch vs scikit-learn & simpan 3 plot margin 2D

## 5. Naive Bayes & TAN — Anggota 5
Komparasi `src.naive_bayes.NaiveBayes` vs `sklearn.naive_bayes.GaussianNB` & Visualisasi Graf DAG Bayesian Network

In [ ]:
from sklearn.naive_bayes import GaussianNB
# TODO (Anggota 5): Evaluasi NaiveBayes / TAN from scratch vs scikit-learn & simpan graf BN